# CNM Effects: RHIC d+Au 200 GeV (Proposed Framework)

This notebook uses the unified `CNMCombine` module to compute and plot Combined Nuclear Matter effects:
1. **nPDF** (EPPS21 + CT18A)
2. **Coherent Energy Loss** (Arleo-Peigné)
3. **Cronin Broadening** ($p_T$ broadening)

It outputs publication-quality figures for $R_{dA}$ vs $y$, $p_T$, and Centrality.

In [1]:
import sys
import numpy as np
from pathlib import Path

# Ensure code is in path
sys.path.append("../cnm_combine")
sys.path.append("../eloss_code")
sys.path.append("../npdf_code")

from cnm_combine import CNMCombine, cnm_vs_y_to_dataframe, cnm_vs_pT_to_dataframe, cnm_vs_cent_to_dataframe
from eloss_cronin_centrality import (
    plot_RpA_vs_y_components_per_centrality,
    plot_RpA_vs_pT_components_per_centrality,
    plot_RpA_vs_centrality_components_band,
    set_publication_style
)

# Apply signature plotting style (Times New Roman, inward ticks, etc.)
set_publication_style()

## 1. Configuration for RHIC 200 GeV

In [2]:
# Define Physics / Binning for RHIC d+Au 200 GeV
comb = CNMCombine.from_defaults(
    energy="200", 
    family="charmonia",
    # Standard RHIC centrality bins
    cent_bins=[(0,20), (20,40), (40,60), (60,100)],
    # Safe rapidity range and windows
    y_edges=np.arange(-2.2, 2.3, 0.2),
    y_windows=[
        (-2.2, -1.2, r"$-2.2 < y < -1.2$ (Backward)"),
        (-0.35, 0.35, r"$|y| < 0.35$ (Mid)"),
        (1.2, 2.2, r"$1.2 < y < 2.2$ (Forward)")
    ],
    pt_range_avg=(0.0, 5.0),
    pt_floor_w=1.0,
    # Physics parameters
    q0_pair=(0.05, 0.09),      # Energy loss transport coeff qhat0
    p0_scale_pair=(0.9, 1.1),  # Broadening scale uncertainty
)

print(f"Initialized {comb.energy} GeV {comb.family} analysis.")
print(f"Target A={comb.gl.spec.A}, Sigma_NN={comb.gl.spec.sigma_nn_mb} mb")

[OpticalGlauber] Target A=197, d=0.535 fm, σ_nn=42.00 mb
[OpticalGlauber] ∫ d²s T_A(s) ≈ 196.792  (should be ~A=197)
[OpticalGlauber] ∫ d²s T_d(s) ≈ 1.9972  (should be ~2)
[OpticalGlauber] Tabulating overlaps T_AA(b), T_pA(b), T_dA(b)...
[OpticalGlauber] σ_tot: AA=7050.0 mb, pA=1699.6 mb, dA=2112.7 mb
Initialized 200 GeV charmonia analysis.
Target A=197, Sigma_NN=42.0 mb


## 2. $R_{dA}$ vs Rapidity

In [3]:
# Compute all components vs y
y_cent, tags_y, cnm_y = comb.cnm_vs_y()

# Determine output directory
OUTDIR = Path("../outputs/RHIC/CNM_200_updated")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Save data
df_y = cnm_vs_y_to_dataframe(y_cent, tags_y, cnm_y, "cnm")
df_y.to_csv(OUTDIR / "r_da_vs_y.csv", index=False)

# Plot (Signature Style)
fig_y, axes_y = plot_RpA_vs_y_components_per_centrality(
    comb.particle, comb.sqrt_sNN, comb.qp_base, comb.gl, comb.cent_bins,
    comb.y_edges, comb.pt_range_avg,
    show_components=("loss", "broad", "total"), # 'total' is eloss*broad, you can add 'npdf' if supported in this plotter helper
    mb_weight_mode="exp",
    suptitle=f"$d+Au$ $\\sqrt{{s_{{NN}}}} = 200$ GeV"
)
fig_y.savefig(OUTDIR / "plot_rda_vs_y.pdf")

KeyError: '0-20%'

## 3. $R_{dA}$ vs $p_T$

In [ ]:
# Compute for the 'Mid' rapidity window example
y0, y1, label = comb.y_windows[1] # Mid

pT_cent, tags_pt, cnm_pt = comb.cnm_vs_pT((y0, y1, label))

# Save data
df_pt = cnm_vs_pT_to_dataframe(pT_cent, tags_pt, cnm_pt, "cnm")
df_pt.to_csv(OUTDIR / "r_da_vs_pT_midrapidity.csv", index=False)

# Plot
fig_pt, axes_pt = plot_RpA_vs_pT_components_per_centrality(
    comb.particle, comb.sqrt_sNN, comb.qp_base, comb.gl, comb.cent_bins,
    comb.p_edges, (y0, y1),
    show_components=("loss", "broad", "total"),
    suptitle=f"$d+Au$ Mid-Rapidity"
)
fig_pt.savefig(OUTDIR / "plot_rda_vs_pT_mid.pdf")

## 4. $R_{dA}$ vs Centrality

In [ ]:
# Calculate integrated RpA vs Centrality for Mid-rapidity
cnm_cent_res = comb.cnm_vs_centrality((y0, y1, label))

# Save
df_cent = cnm_vs_cent_to_dataframe(comb.cent_bins, cnm_cent_res, "cnm")
df_cent.to_csv(OUTDIR / "r_da_vs_cent_mid.csv", index=False)

# Plotting helper requires dictionaries of results
# Unpack 'cnm' components (nPDF*eloss*broad)
# The result structure is (Rc, Rlo, Rhi, Rc_MB, Rlo_MB, Rhi_MB)
rc, rlo, rhi, mb_c, mb_lo, mb_hi = cnm_cent_res["cnm"]

# Map to labels
labels = [f"{int(a)}-{int(b)}%" for (a,b) in comb.cent_bins]
RT_c  = {l: v for l, v in zip(labels, rc)}
RT_lo = {l: v for l, v in zip(labels, rlo)}
RT_hi = {l: v for l, v in zip(labels, rhi)}
RMB_tot = (mb_c, mb_lo, mb_hi)

# Plot
fig_c, ax_c = plot_RpA_vs_centrality_components_band(
    comb.cent_bins, labels,
    RT_c=RT_c, RT_lo=RT_lo, RT_hi=RT_hi, RMB_tot=RMB_tot,
    show=("total",),  # Show the final combined result
    ylabel=r"$R_{dA}$",
    system_label="CNM Total"
)
fig_c.savefig(OUTDIR / "plot_rda_vs_centrality_mid.pdf")